# Multivariate Time-Series Forecasting

## 2. LSTM Model

This notebook develops a Long Short-Term Memory (LSTM) neural network for multivariate time-series demand forecasting.

The LSTM model is designed to learn temporal dependencies from historical retail demand and related explanatory variables.

The model will use a sequence of historical observations to predict the next day's sales.

The same time-based train-validation split used in the baseline and RNN stages will be maintained to ensure a fair comparison.

The LSTM model will be evaluated using:

- MAE
- RMSE
- WAPE

The LSTM performance will be compared against:

- Naive baseline
- Seasonal Naive baseline
- 28-day Moving Average baseline
- RNN

The raw and engineered datasets will remain unchanged.

In [1]:
import gc
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)

Libraries imported successfully.
TensorFlow version: 2.20.0


In [2]:
print("====================================")
print("GPU Check")
print("====================================")

gpus = tf.config.list_physical_devices("GPU")

print("Available GPUs:", gpus)

if gpus:
    print("GPU is available and will be used by TensorFlow.")
else:
    print("WARNING: GPU is not available.")

GPU Check
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU is available and will be used by TensorFlow.


In [3]:
SEQUENCE_LENGTH = 28
FORECAST_HORIZON = 1
TARGET_COLUMN = "sales"

FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

DATA_PATH = "/content/drive/MyDrive/features_event_snap.parquet"

TRAIN_END_DATE = "2016-03-27"
VALIDATION_START_DATE = "2016-03-28"
VALIDATION_END_DATE = "2016-04-24"

BATCH_SIZE = 64
EPOCHS = 10

TRAIN_STEPS = 5000
VALIDATION_STEPS = 500

TEST_SERIES_LIMIT = 1

print("====================================")
print("LSTM Configuration")
print("====================================")

print("Sequence length:", SEQUENCE_LENGTH)
print("Forecast horizon:", FORECAST_HORIZON)
print("Number of features:", len(FEATURE_COLUMNS))
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Training steps:", TRAIN_STEPS)
print("Validation steps:", VALIDATION_STEPS)
print("Training end:", TRAIN_END_DATE)
print("Validation:", VALIDATION_START_DATE, "→", VALIDATION_END_DATE)

LSTM Configuration
Sequence length: 28
Forecast horizon: 1
Number of features: 22
Batch size: 64
Epochs: 10
Training steps: 5000
Validation steps: 500
Training end: 2016-03-27
Validation: 2016-03-28 → 2016-04-24


In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
parquet_file = pq.ParquetFile(DATA_PATH)

print("====================================")
print("Dataset Validation")
print("====================================")

print("Dataset path:", DATA_PATH)
print("Rows:", parquet_file.metadata.num_rows)
print("Columns:", parquet_file.metadata.num_columns)
print("Row groups:", parquet_file.num_row_groups)

print("\nExpected:")
print("Rows: 58,327,370")
print("Columns: 42")
print("Row groups: 61")

print("\nDataset schema:")
print(parquet_file.schema.names)

Dataset Validation
Dataset path: /content/drive/MyDrive/features_event_snap.parquet
Rows: 58327370
Columns: 42
Row groups: 61

Expected:
Rows: 58,327,370
Columns: 42
Row groups: 61

Dataset schema:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'price_change_1', 'price_change_pct_1', 'price_relative_7', 'is_event_day', 'event_count', 'snap_active']


In [7]:
def prepare_features(df):
    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    # Missing price means price was unavailable.
    # price_available explicitly preserves this information.
    df[price_features] = df[price_features].fillna(0)

    # Initial history/rolling values are unavailable
    # because insufficient historical observations exist.
    df[history_features] = df[history_features].fillna(0)

    return df

In [8]:
required_columns = ["item_id", "store_id", "date"] + FEATURE_COLUMNS

test_df = parquet_file.read_row_group(
    0,
    columns=required_columns
).to_pandas()

test_df["date"] = pd.to_datetime(test_df["date"])

print("====================================")
print("Feature Preparation Test")
print("====================================")

print("Rows:", len(test_df))
print("Features:", len(FEATURE_COLUMNS))

print("\nMissing values BEFORE preparation:")
print(test_df[FEATURE_COLUMNS].isna().sum())

test_df = prepare_features(test_df)

print("\nMissing values AFTER preparation:")
print(test_df[FEATURE_COLUMNS].isna().sum().sum())

X_test = test_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)

print("\nFinal feature matrix shape:", X_test.shape)
print("Expected feature count:", len(FEATURE_COLUMNS))

print("\nAny NaN:", np.isnan(X_test).any())
print("Any infinite:", np.isinf(X_test).any())

del test_df, X_test
gc.collect()

Feature Preparation Test
Rows: 1048576
Features: 22

Missing values BEFORE preparation:
sales                      0
sell_price            367871
price_available            0
day_of_month               0
week_of_year               0
day_of_year                0
quarter                    0
is_weekend                 0
lag_1                   1000
lag_7                   7000
lag_14                 14000
lag_28                 28000
rolling_mean_7          7000
rolling_mean_28        28000
rolling_std_7           7000
rolling_std_28         28000
price_change_1        367871
price_change_pct_1    367871
price_relative_7      373905
is_event_day               0
event_count                0
snap_active                0
dtype: int64

Missing values AFTER preparation:
0

Final feature matrix shape: (1048576, 22)
Expected feature count: 22

Any NaN: False
Any infinite: False


9

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

TRAIN_START_DATE = pd.Timestamp("2011-01-29")
TRAIN_END_TIMESTAMP = pd.Timestamp(TRAIN_END_DATE)

required_columns = ["date"] + FEATURE_COLUMNS

total_train_rows = 0

print("====================================")
print("Fitting StandardScaler")
print("====================================")

for rg in range(parquet_file.num_row_groups):

    df = parquet_file.read_row_group(
        rg,
        columns=required_columns
    ).to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    # Keep only training period
    df = df[
        (df["date"] >= TRAIN_START_DATE) &
        (df["date"] <= TRAIN_END_TIMESTAMP)
    ]

    if len(df) == 0:
        del df
        continue

    df = prepare_features(df)

    X = df[FEATURE_COLUMNS].to_numpy(dtype=np.float64)

    scaler.partial_fit(X)

    total_train_rows += len(X)

    del df, X
    gc.collect()

print("\nScaler fitting completed.")

print("Training rows used:", total_train_rows)

print("\nFirst 5 feature means:")
print(scaler.mean_[:5])

print("\nFirst 5 feature standard deviations:")
print(scaler.scale_[:5])

Fitting StandardScaler

Scaler fitting completed.
Training rows used: 57473650

First 5 feature means:
[ 1.12245843  3.4636664   0.7859991  15.71458886 26.0397878 ]

First 5 feature standard deviations:
[ 3.87698193  3.51547202  0.41012744  8.79354866 15.17225613]


In [10]:
# Test the fitted scaler on one sample

sample_df = parquet_file.read_row_group(
    0,
    columns=FEATURE_COLUMNS
).to_pandas()

sample_df = prepare_features(sample_df)

sample_X = sample_df[FEATURE_COLUMNS].iloc[[100]].to_numpy(
    dtype=np.float64
)

scaled_sample = scaler.transform(sample_X)

print("====================================")
print("Scaler Test")
print("====================================")

print("Original shape:", sample_X.shape)
print("Scaled shape:", scaled_sample.shape)

print("\nFirst 10 original values:")
print(sample_X[0, :10])

print("\nFirst 10 scaled values:")
print(scaled_sample[0, :10])

print("\nAny NaN:", np.isnan(scaled_sample).any())
print("Any infinite:", np.isinf(scaled_sample).any())

del sample_df, sample_X, scaled_sample
gc.collect()

Scaler Test
Original shape: (1, 22)
Scaled shape: (1, 22)

First 10 original values:
[  0.   0.   0.   9.  19. 129.   2.   0.   0.   0.]

First 10 scaled values:
[-0.28951861 -0.98526354 -1.91647527 -0.76358125 -0.46399084 -0.47314583
 -0.40831348 -0.63363001 -0.2894102  -0.28867355]

Any NaN: False
Any infinite: False


0

In [15]:
def sequence_generator(
    parquet_file,
    scaler,
    feature_columns,
    sequence_length,
    start_date,
    end_date,
    max_series=None
):
    required_columns = [
        "item_id",
        "store_id",
        "date"
    ] + feature_columns

    start_date = pd.Timestamp(start_date)
    end_date = pd.Timestamp(end_date)

    series_count = 0
    current_key = None
    current_series = []

    for rg in range(parquet_file.num_row_groups):

        df = parquet_file.read_row_group(
            rg,
            columns=required_columns
        ).to_pandas()

        df["date"] = pd.to_datetime(df["date"])

        for key, group in df.groupby(
            ["item_id", "store_id"],
            sort=False
        ):

            group = group.sort_values("date")

            if current_key == key:
                current_series.append(group)

            else:

                if current_key is not None:

                    full_series = pd.concat(
                        current_series,
                        ignore_index=True
                    ).sort_values("date")

                    full_series = full_series[
                        (full_series["date"] >= start_date) &
                        (full_series["date"] <= end_date)
                    ]

                    if len(full_series) > sequence_length:

                        full_series = prepare_features(
                            full_series
                        )

                        X_scaled = scaler.transform(
                            full_series[feature_columns].to_numpy(dtype=np.float64)
                        )

                        for i in range(
                            sequence_length,
                            len(full_series)
                        ):

                            X = X_scaled[
                                i-sequence_length:i
                            ].astype(np.float32)

                            y = np.float32(
                                full_series["sales"].iloc[i]
                            )

                            yield X, y

                current_key = key
                current_series = [group]

                series_count += 1

                if (
                    max_series is not None
                    and series_count >= max_series
                ):
                    break

        del df
        gc.collect()

        if (
            max_series is not None
            and series_count >= max_series
        ):
            break

    # Process final series
    if current_key is not None:

        full_series = pd.concat(
            current_series,
            ignore_index=True
        ).sort_values("date")

        full_series = full_series[
            (full_series["date"] >= start_date) &
            (full_series["date"] <= end_date)
        ]

        if len(full_series) > sequence_length:

            full_series = prepare_features(
                full_series
            )

            X_scaled = scaler.transform(
                full_series[feature_columns].to_numpy(dtype=np.float64)
            )

            for i in range(
                sequence_length,
                len(full_series)
            ):

                X = X_scaled[
                    i-sequence_length:i
                ].astype(np.float32)

                y = np.float32(
                    full_series["sales"].iloc[i]
                )

                yield X, y

In [16]:
sequence_generator_test = sequence_generator(
    parquet_file=parquet_file,
    scaler=scaler,
    feature_columns=FEATURE_COLUMNS,
    sequence_length=SEQUENCE_LENGTH,
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    max_series=1
)

X_seq_test, y_seq_test = next(
    sequence_generator_test
)

print("====================================")
print("LSTM Sequence Generator Test")
print("====================================")

print("X shape:", X_seq_test.shape)

print("y shape:",
      y_seq_test.shape
      if hasattr(y_seq_test, "shape")
      else "scalar")

print("\nExpected X shape:")
print(
    f"({SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})"
)

print("\nTarget value:", y_seq_test)

print("\nAny NaN in X:",
      np.isnan(X_seq_test).any())

print("Any infinite in X:",
      np.isinf(X_seq_test).any())

print("Any NaN in y:",
      np.isnan(y_seq_test).any())

print("\nLast historical sales value:",
      X_seq_test[-1, 0])

print("Target sales value:",
      y_seq_test)

LSTM Sequence Generator Test
X shape: (28, 22)
y shape: ()

Expected X shape:
(28, 22)

Target value: 0.0

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False

Last historical sales value: -0.28951862
Target sales value: 0.0


In [17]:
def make_train_dataset(
    start_date,
    end_date,
    max_series=None,
    batch_size=BATCH_SIZE
):
    output_signature = (
        tf.TensorSpec(
            shape=(
                SEQUENCE_LENGTH,
                len(FEATURE_COLUMNS)
            ),
            dtype=tf.float32
        ),
        tf.TensorSpec(
            shape=(),
            dtype=tf.float32
        )
    )

    dataset = tf.data.Dataset.from_generator(
        lambda: sequence_generator(
            parquet_file=parquet_file,
            scaler=scaler,
            feature_columns=FEATURE_COLUMNS,
            sequence_length=SEQUENCE_LENGTH,
            start_date=start_date,
            end_date=end_date,
            max_series=max_series
        ),
        output_signature=output_signature
    )

    dataset = dataset.batch(batch_size)

    return dataset

In [18]:
train_dataset_test = make_train_dataset(
    start_date=TRAIN_START_DATE,
    end_date=TRAIN_END_DATE,
    max_series=1,
    batch_size=BATCH_SIZE
)

X_train_test, y_train_test = next(
    iter(train_dataset_test)
)

print("====================================")
print("LSTM Training Dataset Test")
print("====================================")

print("X shape:", X_train_test.shape)
print("y shape:", y_train_test.shape)

print("\nExpected X shape:")
print(
    f"({BATCH_SIZE}, {SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})"
)

print("\nExpected y shape:")
print(f"({BATCH_SIZE},)")

print("\nAny NaN in X:",
      tf.reduce_any(
          tf.math.is_nan(X_train_test)
      ).numpy())

print("Any infinite in X:",
      tf.reduce_any(
          tf.math.is_inf(X_train_test)
      ).numpy())

print("Any NaN in y:",
      tf.reduce_any(
          tf.math.is_nan(y_train_test)
      ).numpy())

print("\nFirst 10 targets:")
print(y_train_test[:10].numpy())

LSTM Training Dataset Test
X shape: (64, 28, 22)
y shape: (64,)

Expected X shape:
(64, 28, 22)

Expected y shape:
(64,)

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False

First 10 targets:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [19]:
def validation_sequence_generator(
    parquet_file,
    scaler,
    feature_columns,
    sequence_length,
    validation_start,
    validation_end,
    max_series=None
):
    required_columns = [
        "item_id",
        "store_id",
        "date"
    ] + feature_columns

    validation_start = pd.Timestamp(validation_start)
    validation_end = pd.Timestamp(validation_end)

    # Need 28 days of history before the first validation target
    history_start = (
        validation_start -
        pd.Timedelta(days=sequence_length)
    )

    series_count = 0
    current_key = None
    current_series = []

    for rg in range(parquet_file.num_row_groups):

        df = parquet_file.read_row_group(
            rg,
            columns=required_columns
        ).to_pandas()

        df["date"] = pd.to_datetime(df["date"])

        df = df[
            (df["date"] >= history_start) &
            (df["date"] <= validation_end)
        ]

        for key, group in df.groupby(
            ["item_id", "store_id"],
            sort=False
        ):

            group = group.sort_values("date")

            if current_key == key:
                current_series.append(group)

            else:

                if current_key is not None:

                    full_series = pd.concat(
                        current_series,
                        ignore_index=True
                    ).sort_values("date")

                    full_series = prepare_features(
                        full_series
                    )

                    X_scaled = scaler.transform(
                        full_series[feature_columns]
                        .to_numpy(dtype=np.float64)
                    )

                    dates = (
                        full_series["date"]
                        .reset_index(drop=True)
                    )

                    for i in range(
                        sequence_length,
                        len(full_series)
                    ):

                        target_date = dates.iloc[i]

                        if (
                            target_date >= validation_start
                            and
                            target_date <= validation_end
                        ):

                            X = X_scaled[
                                i-sequence_length:i
                            ].astype(np.float32)

                            y = np.float32(
                                full_series["sales"].iloc[i]
                            )

                            yield X, y

                current_key = key
                current_series = [group]

                series_count += 1

                if (
                    max_series is not None
                    and series_count >= max_series
                ):
                    break

        del df
        gc.collect()

        if (
            max_series is not None
            and series_count >= max_series
        ):
            break

    # Process final series
    if current_key is not None:

        full_series = pd.concat(
            current_series,
            ignore_index=True
        ).sort_values("date")

        full_series = prepare_features(
            full_series
        )

        X_scaled = scaler.transform(
            full_series[feature_columns]
            .to_numpy(dtype=np.float64)
        )

        dates = (
            full_series["date"]
            .reset_index(drop=True)
        )

        for i in range(
            sequence_length,
            len(full_series)
        ):

            target_date = dates.iloc[i]

            if (
                target_date >= validation_start
                and
                target_date <= validation_end
            ):

                X = X_scaled[
                    i-sequence_length:i
                ].astype(np.float32)

                y = np.float32(
                    full_series["sales"].iloc[i]
                )

                yield X, y

In [20]:
validation_generator_test = validation_sequence_generator(
    parquet_file=parquet_file,
    scaler=scaler,
    feature_columns=FEATURE_COLUMNS,
    sequence_length=SEQUENCE_LENGTH,
    validation_start=VALIDATION_START_DATE,
    validation_end=VALIDATION_END_DATE,
    max_series=1
)

X_val_test, y_val_test = next(
    validation_generator_test
)

print("====================================")
print("LSTM Validation Generator Test")
print("====================================")

print("X shape:", X_val_test.shape)

print(
    "y shape:",
    y_val_test.shape
    if hasattr(y_val_test, "shape")
    else "scalar"
)

print("\nExpected X shape:")
print(
    f"({SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})"
)

print("\nTarget value:", y_val_test)

print("\nAny NaN in X:",
      np.isnan(X_val_test).any())

print("Any infinite in X:",
      np.isinf(X_val_test).any())

print("Any NaN in y:",
      np.isnan(y_val_test).any())

print("\nFirst validation target date:")
print(VALIDATION_START_DATE)

LSTM Validation Generator Test
X shape: (28, 22)
y shape: ()

Expected X shape:
(28, 22)

Target value: 1.0

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False

First validation target date:
2016-03-28


In [21]:
def make_validation_dataset(
    validation_start,
    validation_end,
    max_series=None,
    batch_size=BATCH_SIZE
):
    output_signature = (
        tf.TensorSpec(
            shape=(
                SEQUENCE_LENGTH,
                len(FEATURE_COLUMNS)
            ),
            dtype=tf.float32
        ),
        tf.TensorSpec(
            shape=(),
            dtype=tf.float32
        )
    )

    dataset = tf.data.Dataset.from_generator(
        lambda: validation_sequence_generator(
            parquet_file=parquet_file,
            scaler=scaler,
            feature_columns=FEATURE_COLUMNS,
            sequence_length=SEQUENCE_LENGTH,
            validation_start=validation_start,
            validation_end=validation_end,
            max_series=max_series
        ),
        output_signature=output_signature
    )

    dataset = dataset.batch(batch_size)

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset

In [22]:
validation_dataset_test = make_validation_dataset(
    validation_start=VALIDATION_START_DATE,
    validation_end=VALIDATION_END_DATE,
    max_series=1,
    batch_size=BATCH_SIZE
)

X_val_batch, y_val_batch = next(
    iter(validation_dataset_test)
)

print("====================================")
print("LSTM Validation Dataset Test")
print("====================================")

print("X shape:", X_val_batch.shape)
print("y shape:", y_val_batch.shape)

print("\nExpected X shape:")
print(
    f"({BATCH_SIZE}, {SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})"
)

print("\nExpected y shape:")
print(f"({BATCH_SIZE},)")

print("\nAny NaN in X:",
      tf.reduce_any(
          tf.math.is_nan(X_val_batch)
      ).numpy())

print("Any infinite in X:",
      tf.reduce_any(
          tf.math.is_inf(X_val_batch)
      ).numpy())

print("Any NaN in y:",
      tf.reduce_any(
          tf.math.is_nan(y_val_batch)
      ).numpy())

print("\nFirst 10 validation targets:")
print(y_val_batch[:10].numpy())

LSTM Validation Dataset Test
X shape: (28, 28, 22)
y shape: (28,)

Expected X shape:
(64, 28, 22)

Expected y shape:
(64,)

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [23]:
validation_dataset = make_validation_dataset(
    validation_start=VALIDATION_START_DATE,
    validation_end=VALIDATION_END_DATE,
    max_series=None,
    batch_size=BATCH_SIZE
)

print("====================================")
print("Final LSTM Validation Dataset")
print("====================================")

print("Batch size:", BATCH_SIZE)

print("Validation period:")
print(
    VALIDATION_START_DATE,
    "→",
    VALIDATION_END_DATE
)

print("Expected validation observations:")
print(30490 * 28)

print("Shuffle: disabled")
print("Prefetch: enabled")

Final LSTM Validation Dataset
Batch size: 64
Validation period:
2016-03-28 → 2016-04-24
Expected validation observations:
853720
Shuffle: disabled
Prefetch: enabled


In [24]:
X_val_batch, y_val_batch = next(
    iter(validation_dataset)
)

print("====================================")
print("Final LSTM Validation Batch Test")
print("====================================")

print("X shape:", X_val_batch.shape)
print("y shape:", y_val_batch.shape)

print("\nExpected X shape:")
print(
    f"({BATCH_SIZE}, {SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})"
)

print("\nExpected y shape:")
print(f"({BATCH_SIZE},)")

print("\nAny NaN in X:",
      tf.reduce_any(
          tf.math.is_nan(X_val_batch)
      ).numpy())

print("Any infinite in X:",
      tf.reduce_any(
          tf.math.is_inf(X_val_batch)
      ).numpy())

print("Any NaN in y:",
      tf.reduce_any(
          tf.math.is_nan(y_val_batch)
      ).numpy())

print("Any infinite in y:",
      tf.reduce_any(
          tf.math.is_inf(y_val_batch)
      ).numpy())

print("\nFirst 10 validation targets:")
print(y_val_batch[:10].numpy())

Final LSTM Validation Batch Test
X shape: (64, 28, 22)
y shape: (64,)

Expected X shape:
(64, 28, 22)

Expected y shape:
(64,)

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False
Any infinite in y: False

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [25]:
model = Sequential([
    Input(
        shape=(
            SEQUENCE_LENGTH,
            len(FEATURE_COLUMNS)
        )
    ),

    LSTM(
        64,
        activation="tanh"
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

print("====================================")
print("LSTM Model")
print("====================================")

model.summary()

LSTM Model


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        22,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,385 (95.25 KB)

 Trainable params: 24,385 (95.25 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
train_dataset = make_train_dataset(
    start_date=TRAIN_START_DATE,
    end_date=TRAIN_END_DATE,
    max_series=None,
    batch_size=BATCH_SIZE
)

train_dataset = train_dataset.shuffle(
    buffer_size=4096,
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

print("====================================")
print("Final LSTM Training Dataset")
print("====================================")

print("Training period:")
print(
    TRAIN_START_DATE,
    "→",
    TRAIN_END_DATE
)

print("Batch size:", BATCH_SIZE)
print("Training steps:", TRAIN_STEPS)
print("Shuffle buffer:", 4096)
print("Prefetch: enabled")

Final LSTM Training Dataset
Training period:
2011-01-29 00:00:00 → 2016-03-27
Batch size: 64
Training steps: 5000
Shuffle buffer: 4096
Prefetch: enabled


In [27]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_data=validation_dataset,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1
)

print("\n====================================")
print("LSTM Training Completed")
print("====================================")

print("Epochs completed:", len(history.history["loss"]))

print(
    "Best validation loss:",
    min(history.history["val_loss"])
)

Epoch 1/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 254s 32ms/step - loss: 6.4007 - mae: 0.8131 - val_loss: 4.2481 - val_mae: 1.0555
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 158s 32ms/step - loss: 3.5338 - mae: 0.6474 - val_loss: 3.8368 - val_mae: 0.9597
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 158s 32ms/step - loss: 2.4552 - mae: 0.6842 - val_loss: 3.8497 - val_mae: 1.0045
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 157s 31ms/step - loss: 4.2538 - mae: 0.8377 - val_loss: 3.8305 - val_mae: 1.0020
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 159s 32ms/step - loss: 2.8188 - mae: 0.8314 - val_loss: 3.7218 - val_mae: 0.9666
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 156s 31ms/step - loss: 3.3395 - mae: 0.9014 - val_loss: 3.7416 - val_mae: 0.9823
Epoch 7/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 158s 32ms/step - loss: 2.1784 - mae: 0.7121 - val_loss: 3.7577 - val_mae: 0.9559

LSTM Training Completed
Epochs completed: 7
Best validation loss: 3.7218494415283203


In [28]:
# Generate LSTM predictions on the complete validation period

y_true = []
y_pred = []

for X_batch, y_batch in validation_dataset:

    predictions = model.predict(
        X_batch,
        verbose=0
    )

    y_true.append(
        y_batch.numpy()
    )

    y_pred.append(
        predictions.squeeze()
    )

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

print("====================================")
print("LSTM Validation Predictions")
print("====================================")

print("Actual values shape:", y_true.shape)
print("Predicted values shape:", y_pred.shape)

print("\nExpected validation observations:")
print(30490 * 28)

print("\nAny NaN in predictions:",
      np.isnan(y_pred).any())

print("Any infinite in predictions:",
      np.isinf(y_pred).any())

print("\nFirst 10 actual values:")
print(y_true[:10])

print("\nFirst 10 predictions:")
print(y_pred[:10])

LSTM Validation Predictions
Actual values shape: (853720,)
Predicted values shape: (853720,)

Expected validation observations:
853720

Any NaN in predictions: False
Any infinite in predictions: False

First 10 actual values:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]

First 10 predictions:
[1.1474766  1.0803883  0.9186208  0.90269315 1.0001342  1.0499833
 1.1228936  0.9739095  0.732001   0.83584654]


In [29]:
# Calculate LSTM validation metrics

mae = np.mean(
    np.abs(y_true - y_pred)
)

rmse = np.sqrt(
    np.mean(
        (y_true - y_pred) ** 2
    )
)

wape = (
    np.sum(
        np.abs(y_true - y_pred)
    )
    /
    np.sum(
        np.abs(y_true)
    )
) * 100

print("====================================")
print("LSTM Validation Performance")
print("====================================")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.4f}%")

LSTM Validation Performance
MAE  : 1.0220
RMSE : 2.3766
WAPE : 73.7131%


## 6. LSTM Results and Findings

The LSTM model was evaluated on the same 28-day time-based validation period used for the baseline and RNN models.

### Validation Period

- Training period: 2011-01-29 to 2016-03-27
- Validation period: 2016-03-28 to 2016-04-24
- Validation observations: 853,720
- Item-store series: 30,490

### Model Performance

| Model | MAE | RMSE | WAPE |
|---|---:|---:|---:|
| Naive | 1.1798 | 2.5948 | 85.10% |
| Seasonal Naive (7-day) | 1.2054 | 2.6601 | 86.94% |
| 28-day Moving Average | **1.0050** | **2.0935** | **72.49%** |
| RNN | 1.0326 | 2.4927 | 74.48% |
| LSTM | 1.0220 | 2.3766 | 73.71% |

### Key Findings

The LSTM model improved upon the RNN across MAE, RMSE, and WAPE.

The LSTM achieved a WAPE of 73.71%, compared with 74.48% for the RNN.

However, the 28-day Moving Average remained the strongest model among the evaluated approaches, achieving a WAPE of 72.49%.

Therefore, the current LSTM configuration improves upon the RNN but does not yet outperform the strongest baseline.

No additional LSTM tuning was performed at this stage so that subsequent models can be evaluated under a consistent experimental framework.

## 7. Conclusion

The LSTM forecasting stage has been completed successfully.

The model used 28 days of historical observations and 22 multivariate features to predict next-day demand.

The LSTM achieved:

- MAE: **1.0220**
- RMSE: **2.3766**
- WAPE: **73.71%**

The LSTM outperformed the RNN across all three evaluation metrics.

However, the 28-day Moving Average baseline remained stronger, with a WAPE of **72.49%**.

The LSTM results will be retained as a benchmark for comparison with the subsequent GRU, LSTM with Attention, and Transformer models.

The LSTM stage is complete.